In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install hqq
!pip install --upgrade bitsandbytes transformers peft accelerate datasets trl
!pip install wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb
user_secrets = UserSecretsClient()
token=user_secrets.get_secret("HUGGINGFACE_API_KEY")
secret_value_1 = user_secrets.get_secret("WANDB_LOGIN_API")
os.environ["WANDB_API_KEY"] = secret_value_1

# Log in to WandB
wandb.login()

In [ ]:
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    HqqConfig
)
from trl import SFTTrainer
import torch, multiprocessing

model_id = "microsoft/Phi-3.5-mini-instruct"
# All linear layers will use the same quantization config
quant_config = HqqConfig(nbits=1, group_size=64, quant_zero=False, quant_scale=False, axis=1)

#Load the tokenizer to save it along with the model
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,token=token)
tokenizer.pad_token = tokenizer.eos_token

# Load and quantize
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=quant_config,
    token=token
)

from hqq.models.hf.base import AutoHQQHFModel

#Save: Make sure to save the model BEFORE any patching
AutoHQQHFModel.save_quantized(model, "./Phi-3.5-mini-instruct-1bit/")

model = prepare_model_for_kbit_training(model)



In [ ]:
from transformers import TrainerCallback
import time

class TimerCallback(TrainerCallback):
    def __init__(self, save_time_in_seconds, trainer):
        self.start_time = None
        self.save_time_in_seconds = save_time_in_seconds
        self.trainer = trainer

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed_time = time.time() - self.start_time
        if elapsed_time > self.save_time_in_seconds:
            # Save the model checkpoint
            self.trainer.save_model(args.output_dir)
            self.trainer.log({"info": "Checkpoint saved after 9 hours"})
            # Reset the timer
            self.start_time = time.time()
nine_hours_in_seconds = 9 * 60 * 60

In [ ]:
dataset = load_dataset("timdettmers/openassistant-guanaco")

peft_config = LoraConfig(
        lora_alpha=16,
        lora_dropout=0.05,
        r=16,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules= ['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"]
)



#Add the EOS token
def process(row):
    row["text"] = row["text"]+"<|end_of_text|>"
    return row

ds = dataset.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)
from trl import SFTConfig

training_arguments = SFTConfig(
        output_dir="./Phi-3.5-mini-instruct-1bit-openassistant-guanaco/",
        evaluation_strategy="steps",
        do_eval=True,
        optim="paged_adamw_8bit",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=8,
        log_level="debug",
        save_strategy="epoch",
        logging_steps=100,
        learning_rate=1e-4,
        eval_steps=100,
        num_train_epochs=3,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_arguments,
)

# Add the timer callback
trainer.add_callback(TimerCallback(save_time_in_seconds=nine_hours_in_seconds, trainer=trainer))

trainer.train()

In [ ]:
import os

def get_dir_size_mb(path):
    total_bytes = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_bytes += os.path.getsize(fp)
    return total_bytes / (1024 * 1024)

quant_dir = "./Phi-3.5-mini-instruct-1bit/"
adapter_dir = training_arguments.output_dir

# microsoft/Phi-3.5-mini-instruct has ~3.82B parameters; baseline stored in bf16 (2 bytes/param)
baseline_size_mb = (3.82 * 10**9 * 2) / (1024 * 1024)
quantized_size_mb = get_dir_size_mb(quant_dir)
adapter_size_mb = get_dir_size_mb(adapter_dir) if os.path.isdir(adapter_dir) else 0.0

reduction_pct = ((baseline_size_mb - quantized_size_mb) / baseline_size_mb) * 100

print(f"Estimated Baseline (BF16) Size : {baseline_size_mb:.2f} MB (~{baseline_size_mb / 1024:.2f} GB)")
print(f"Quantized (1-bit HQQ) Size     : {quantized_size_mb:.2f} MB (~{quantized_size_mb / 1024:.2f} GB)")
print(f"LoRA Adapter Size              : {adapter_size_mb:.2f} MB")
print(f"Weight Footprint Reduction     : {reduction_pct:.2f}%")

if wandb.run is not None:
    wandb.log({
        "footprint/baseline_mb": baseline_size_mb,
        "footprint/quantized_mb": quantized_size_mb,
        "footprint/adapter_mb": adapter_size_mb,
        "footprint/reduction_pct": reduction_pct,
    })

In [ ]:
import torch
from datasets import load_dataset
from tqdm import tqdm

def calculate_perplexity(model, tokenizer, test_dataset, max_length=512, stride=256):
    model.eval()
    device = next(model.parameters()).device

    encodings = tokenizer("\n\n".join(test_dataset["text"]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0

    for begin_loc in tqdm(range(0, seq_len, stride), desc="Evaluating Perplexity"):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # tokens we calculate loss on
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100  # mask context tokens

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss  # mean loss per active token

        nlls.append(neg_log_likelihood * trg_len)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / seq_len)
    return ppl.item()

# Load WikiText-2 test set
test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Compute PPL on the fine-tuned 1-bit model
ppl_score = calculate_perplexity(model, tokenizer, test_data)
print(f"Fine-tuned 1-bit Model Perplexity (WikiText-2): {ppl_score:.2f}")

if wandb.run is not None:
    wandb.log({"eval/perplexity_wikitext2": ppl_score})

In [ ]:
import time

def benchmark_inference(model, tokenizer, prompt="Explain quantum computing in simple terms:", max_new_tokens=64):
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Warm-up run
    with torch.no_grad():
        _ = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
        )

    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed_time = time.time() - start_time

    generated_tokens = output.shape[1] - inputs.input_ids.shape[1]
    throughput = generated_tokens / elapsed_time
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024) if device.type == "cuda" else 0.0

    print(f"Generated text: {tokenizer.decode(output[0], skip_special_tokens=True)}")
    print("-" * 50)
    print(f"Tokens Generated     : {generated_tokens}")
    print(f"Elapsed Time         : {elapsed_time:.3f} s")
    print(f"Inference Throughput : {throughput:.2f} tokens/s")
    print(f"Peak VRAM Usage      : {peak_vram_mb:.2f} MB")

    if wandb.run is not None:
        wandb.log({
            "latency/throughput_tokens_per_sec": throughput,
            "latency/peak_vram_mb": peak_vram_mb,
        })

benchmark_inference(model, tokenizer)